# RK4 Metodoa eta Urrats Luzera Handiak: Ilargiaren Etenguneen Eragina

Notebook honetan **RK4 (4. ordenako)** metodoa aztertuko dugu urrats luzera handiekin.
Helburua da ikustea nola efemerideen "etenguneek" edo nodoek ($t_{knot}$) eragina izan dezaketen zehaztasunean.

Helburua:
1.  Simulazioa hasieratik (URT 1) **Lurretik gertueneko pasuarteraino ($t_{CA}$)** eraman.
2.  Puntu horretatik gertu dagoen **Ilargiaren etengune bat ($t_{knot}$)** bilatu eta hortik hasi analisia.
3.  Bi kasu konparatu $h$ (pausoa) desberdinekin:
    *   **Kasu A (Sinkronizatu gabe)**: $h$ arbitrarioa.
    *   **Kasu B (Sinkronizatua)**: $h$-k zehazki zatitzen du etenguneen arteko tartea ($L_{MOON}$).
4.  Errorea aztertu eta $O(h^{4})$ proportzioa mantentzen den ikusi.

In [ ]:
using Revise
using Pkg; Pkg.activate("..")
using GravitationSimulation
using LittleEphemeris
using JSON
using CSV
using SPICE
using DataFrames
using LinearAlgebra
using Plots
# using IRKGaussLegendre # Ez dugu hau erabiliko orain
import SciMLBase

  Activating project at `~/Documentos/Ingenieritza_Informatikoa/GrAL/GrAL_github`


In [2]:
# Karga nukleoak
furnsh("./data/naif0012.tls", "./data/de440.bsp")

In [3]:
# Gorputzen koefizienteak kargatu eta prestatu
ID_list = [1, 2, 3, 4, 5, 6, 7, 8, 10, 301]

# Denbora tartea: Urtarrila +
et_0 = 10593.535998938605 * 86400
et_end = et_0 + (150 * 86400) # Apirila/Maiatza arte nahikoa da

time_interval = (et_0, et_end)
time_interval_list = fill(time_interval, length(ID_list))

create_coeffs_file("./data/coeffs.json", "./data/coeffs.csv", ID_list, time_interval_list)

Mercury = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 1, time_interval);
Venus   = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 2, time_interval);
Earth   = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 3, time_interval);
Mars    = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 4, time_interval);
Jupiter = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 5, time_interval);
Saturn  = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 6, time_interval);
Uranus  = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 7, time_interval);
Neptune = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 8, time_interval);
Sun     = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 10, time_interval);
Moon    = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 301, time_interval);

In [5]:
# Hasierako balioak eta konstanteak
u0 = [-5.5946538550488512E-01, 8.5647564757574512E-01, 3.0415066217102493E-01,
      -1.3818324735921638E-02, -6.0088275597939191E-03, -2.5805044631309632E-03]

# Heliozentrikotik barizentrikora
eguzkia_pos_barizentrikoa = [0.001232781221250307, -0.0012750764430978325, -0.0005187131180711941]
u0[1:3] += eguzkia_pos_barizentrikoa

# Unit conversion
AU = 149597870.7
DAY = 86400.0
u0[1:3] *= AU
u0[4:6] *= (AU / DAY)

# Grabitazio parametroak (km^3/s^2)
mu_S = 1.32712440042e11
mu_E = 398600.435
mu_M = 4902.80
mu_Mer = 22032.09
mu_V = 324858.59
mu_Ma = 42828.37
mu_J = 1.26686534e8
mu_Sat = 3.7931187e7
mu_U = 5.793939e6
mu_N = 6.836529e6

p_all = [mu_S, mu_E, mu_M, mu_Mer, mu_V, mu_Ma, mu_J, mu_Sat, mu_U, mu_N]
# Lehen agian p_all = [mu1, mu2, ...] zenuen. 
# Orain egitura hau behar du:
p_all = (
    mus = [mu_S, mu_E, mu_M, mu_Mer, mu_V, mu_Ma, mu_J, mu_Sat, mu_U, mu_N], # mu-en zerrenda
    bodies = [
        Sun, Earth, Moon, Mercury,
        Venus, Mars, Jupiter, Saturn, Uranus, Neptune] # posizioen zerrenda
)

t_0 = et_0
t_end = et_end

N_samples = 1000
dt_total = t_end - t_0
dt_out = dt_total / (N_samples - 1)

println("Simulazio parametroak:")
println("Hasiera (ET): $t_0")
println("Bukaera (ET): $t_end")
println("Lagin kopurua (Nomatua): $N_samples")
println("Irteera pausoa (dt_out): $dt_out s (~$(round(dt_out/3600, digits=2)) ordu)")

Simulazio parametroak:
Hasiera (ET): 9.152815103082955e8
Bukaera (ET): 9.282415103082955e8
Lagin kopurua (Nomatua): 1000
Irteera pausoa (dt_out): 12972.972972972973 s (~3.6 ordu)


In [7]:
# 1. URRATSA: Hasieratik Earth Close Approach-era gerturatu
# ------------------------------------------------------------------
# Integrazio bat egingo dugu Lurretik gertu pasatzen den unera arte.
# Apophis 2029ko Apirilaren 13an pasatzen da.

# Hasierako hurbilketa bat egingo dugu 4 hilabete inguru integratuz.
t_approx_ca = et_0 + (110 * 86400) # Apirilaren erdialdera

# Zehaztasun handiarekin iritsi nahi dugu puntu horretara
dt_pre = 3600.0 # Ordu bat
# RK4 erabiltzen dugu zuzenean
tt_pre, uu_pre = RK4(u0, et_0, t_approx_ca, dt_pre, p_all, f_all!)

u_pre_ca = uu_pre[end]
t_pre_ca = tt_pre[end]

println("Lurretik gertu kokatu gara (t = $t_pre_ca)")

Lurretik gertu kokatu gara (t = 9.247855103082955e8)


In [8]:
# 2. URRATSA: Ilargiaren Etengunea Bilatu
# ------------------------------------------------------------------
# t_pre_ca-tik aurrera dagoen lehen ilargi-nodoa (polinomio muga) bilatuko dugu.
# Hortik aurrera hasiko dugu benetako analisia.

coeffs_data = JSON.parsefile("./data/coeffs.json")
moon_entry = filter(x -> x["bodyID"] == 301, coeffs_data)[1]
intervals = moon_entry["timeIntervals"]

# Bilatu t_pre_ca ondorengo lehen etengunea
idx = findfirst(x -> x > t_pre_ca, intervals)
t_start_analysis = intervals[idx]
L_MOON = intervals[2] - intervals[1] # Polinomio baten iraupena

print("Analisi hasiera puntua (Ilargiaren nodoa): $t_start_analysis\n")
print("Ilargiaren polinomio luzera (L_MOON): $L_MOON s\n")

# Puntu horretara zehazki iristeko integratu berriz
# RK4 erabiltzen dugu tarte txiki hau betetzeko
h_bridge = (t_start_analysis - t_pre_ca) / 100 
tt_bridge, uu_bridge = RK4(u_pre_ca, t_pre_ca, t_start_analysis, h_bridge, p_all, f_all!)

u_start_analysis = uu_bridge[end]
println("Hasierako egoera (u) nodoan kalkulatuta.")

Analisi hasiera puntua (Ilargiaren nodoa): 9.25128e8
Ilargiaren polinomio luzera (L_MOON): 345600.0 s
Hasierako egoera (u) nodoan kalkulatuta.


In [9]:
# 3. URRATSA: Analisiaren Konfigurazioa
# ------------------------------------------------------------------
# Hemendik aurrera, epe labur batean (adib. 10 egun) integratuko dugu
# pauso handiekin, erroreak aztertzeko.

analysis_duration = 30 * 86400.0 # 30 egun
t_end_analysis = t_start_analysis + analysis_duration

println("Analisi iraupena: $(analysis_duration/86400) egun")

# SORTU "EGIA" (Erreferentzia)
# Pauso oso txiki batekin integratuko dugu sinkronizatuta
m_ref = 64
h_ref = L_MOON / m_ref # Polinomio bakoitzeko 64 pauso

# RK4 deia: m=m_ref erabiliz, irteera bakoitzeko m_ref pauso emango ditu
# Horrela tt_ref-en puntuak polinomioaren mugetan egongo dira (L_MOON tarteka)
@time tt_ref, uu_ref = RK4(u_start_analysis, t_start_analysis, t_end_analysis, h_ref, p_all, f_all!, m_ref)

# Soluzioaren azken puntua hartuko dugu oinarri
u_true_end = uu_ref[end]
println("Erreferentzia ("*string(m_ref)*" pauso/polinomio) kalkulatuta.")

Analisi iraupena: 30.0 egun
  0.008379 seconds (143.39 k allocations: 21.690 MiB)
Erreferentzia (64 pauso/polinomio) kalkulatuta.


In [ ]:
# 4. URRATSA: Bi Kasuen Konparaketa
# ------------------------------------------------------------------

# KASU A: Sinkronizatu gabe
# h balioak arbitrarioak dira (ez dute L_MOON zatitzen)
# Adibidez: h = 1 egun, 0.5 egun, ... baina 'zikinak'
ms_no_sync = [4.5, 9.5, 18.5, 36.5] # Zatitzaile ez osoak lortzeko
hs_no_sync = L_MOON ./ ms_no_sync 

# KASU B: Sinkronizatua (NODOEKIN BAT)
# h balioak L_MOON-en zatitzaile zehatzak dira
ms_sync = [4, 8, 16, 32, 64] # 4, 2, 1, 0.5 egun...
hs_sync = L_MOON ./ ms_sync

# Emaitzak gordetzeko
res_no_sync = []
res_sync = []

# --- KASU A exekuzioa ---
println("\n--- KASU A: Sinkronizatu gabe ---")
for h_target in hs_no_sync
    # RK4 funtzioan h finkoa da.
    # Tartera iristeko pauso kopurua kalkulatu
    # Garrantzitsua: t_end_analysis-era iristeko zehazki doitu behar da h, 
    # bestela ez gara puntu berera iristen eta errorea faltsua izango da.
    
    steps = round(Int, analysis_duration / h_target)
    h_actual = analysis_duration / steps
    
    tt, uu = RK4(u_start_analysis, t_start_analysis, t_end_analysis, h_actual, p_all, f_all!)
    err = norm(uu[end][1:3] - u_true_end[1:3]) / norm(u_true_end[1:3])
    
    push!(res_no_sync, (h_actual, err))
    println("h ≈ $(round(h_actual, digits=1)) s | Errorea: $err")
end

# --- KASU B exekuzioa ---
println("\n--- KASU B: Sinkronizatua (Etenguneekin bat) ---")
for h in hs_sync
    # Hemen h zehatza da eta L_MOON zatitzen du.
    # gainera analysis_duration ere L_MOON-en multiploa denez, arazorik gabe iritsiko da.
    
    tt, uu = RK4(u_start_analysis, t_start_analysis, t_end_analysis, h, p_all, f_all!)
    err = norm(uu[end][1:3] - u_true_end[1:3]) / norm(u_true_end[1:3])
    
    push!(res_sync, (h, err))
    println("h = $(round(h, digits=1)) s | Errorea: $err")
end

In [ ]:
# 5. URRATSA: Grafikoak eta Malda
# ------------------------------------------------------------------

h_A = [x[1] for x in res_no_sync]
e_A = [x[2] for x in res_no_sync]

h_B = [x[1] for x in res_sync]
e_B = [x[2] for x in res_sync]

plot(h_A, e_A, xaxis=:log, yaxis=:log, marker=:star, label="Kasu A: Ez sinkronizatua", 
     color=:red, title="RK4: Trunkatze Errorea (Sinkronizazioa)",
     xlabel="Pausoa h (s)", ylabel="Errore Erlatiboa", legend=:bottomright)
plot!(h_B, e_B, xaxis=:log, yaxis=:log, marker=:circle, label="Kasu B: Sinkronizatua", color=:blue)

# O(h^4) lerroa Kasu B-rentzat (RK4 delako orain)
if length(h_B) > 0
    C = e_B[end] / (h_B[end]^4)
    ref_line = [C * h^4 for h in h_B]
    plot!(h_B, ref_line, linestyle=:dash, label="O(h^4)", color=:black)
end

### Ondorioak

Grafikoan ikusi beharko litzateke:
1.  Sinkronizazioarekin (**Kasu B**), errorea $O(h^{4})$ proportzio garbiagoa dela.
2.  Sinkronizazio gabe (**Kasu A**), zarata edo errore handiagoak ager daitezkeela nodoak zeharkatzean.

Honek frogatzen du RK4 bezalako metodo klasikoetan ere, doitasun handian lan egitean, efemerideen egitura kontuan hartzea garrantzitsua dela.